# drone-2025-reposed

# Imports

In [ ]:
import json
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from uavloc.data.annotated import load_label_map
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.datasets import data_root, verify_dataset, write_manifest
from uavloc.geo import LatLon, footprint_m
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H
from uavloc.track import REPO_ROOT

# Params

In [ ]:
NAME = "drone-2025-reposed"
VERSION = "1"
SOURCE_FRAMES = "drone-2025-easy"
CORRECTIONS = "pocs/pose-reregistration/results.csv"

FOV_W_DEG = 60.0                  # correlator geometry; the PoC's scale is relative to this footprint
SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)
LABELS = ("gonchary-2025", "ortho", "ua-ortho-annotated")
PREVIEW_FRAMES = ["drone-1743863832000", "drone-1743863874000", "drone-1743863859000", "drone-1743864262500"]

# Data

In [ ]:
frames = load_frames(SOURCE_FRAMES)
corrections = pd.read_csv(REPO_ROOT / CORRECTIONS).set_index("frame")
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
label_map = load_label_map(*LABELS)
out = data_root(NAME, create=True)
len(frames), int(corrections["registered"].sum())

# Build

In [ ]:
def reposed_meta(frame: DroneFrame, correction: pd.Series) -> dict:
    """Source meta with corrected heading and footprint; position stays as logged."""
    meta = json.loads((frame.image_path.parent / "meta.json").read_text())
    scale = float(correction["scale"])
    return {
        **meta,
        "yaw_deg": float(correction["corrected_yaw_deg"]),
        "alt_m": meta["alt_m"] * scale,
        "mpp": meta["mpp"] * scale,
        "logged_yaw_deg": meta["yaw_deg"],
        "logged_alt_m": meta["alt_m"],
        "yaw_offset_deg": float(correction["yaw_offset_deg"]),
        "footprint_scale": scale,
        "block": int(correction["block"]),
        "ambiguity_m": float(correction["ambiguity_m"]),
        "position_note": "logged; the PoC's corrected position is ambiguous along the track, not used",
    }

In [ ]:
def write_sample(frame: DroneFrame, meta: dict) -> dict:
    """frame.jpg, meta.json, and the orthophoto + label crops at the reposed pose (logged position)."""
    folder = out / frame.name
    folder.mkdir(exist_ok=True)
    shutil.copy2(frame.image_path, folder / "frame.jpg")
    w_m, h_m = footprint_m(meta["alt_m"], FOV_W_DEG, CAMERA_ASPECT_W_OVER_H)
    sat = satellite.crop_rotated(frame.position, meta["yaw_deg"], w_m, h_m)
    label = label_map.crop_rotated(frame.position, meta["yaw_deg"], w_m, h_m, out_mpp=satellite.mpp, nearest=True)
    Image.fromarray(sat).save(folder / "satellite.png")
    Image.fromarray(label).save(folder / "label.png")
    meta = {**meta, "footprint_m": [w_m, h_m], "crop_mpp": satellite.mpp}
    (folder / "meta.json").write_text(json.dumps(meta, indent=1))
    return {"frame": frame.name, "block": meta["block"], "labelled_frac": float((label > 0).mean()),
            "sat_px": sat.shape[:2], "label_px": label.shape[:2]}

In [ ]:
samples = pd.DataFrame([write_sample(f, reposed_meta(f, corrections.loc[f.name])) for f in frames])
samples.groupby("block")["labelled_frac"].agg(["count", "min", "median"])

# Checks

In [ ]:
reposed = load_frames(NAME)
logged = {f.name: f for f in frames}
position_changed_deg = max(abs(f.position.lat - logged[f.name].position.lat) + abs(f.position.lon - logged[f.name].position.lon) for f in reposed)
shapes_match = bool((samples["sat_px"] == samples["label_px"]).all())
checks = {
    "frames": len(reposed),
    "position_changed_deg": position_changed_deg,
    "sat_label_shapes_match": shapes_match,
    "median_abs_yaw_offset_deg": float(corrections["yaw_offset_deg"].abs().median()),
    "median_footprint_scale": float(corrections["scale"].median()),
    "median_ambiguity_m": float(corrections["ambiguity_m"].median()),
    "frames_with_no_label_px": int((samples["labelled_frac"] == 0).sum()),
    "median_labelled_frac": float(samples["labelled_frac"].median()),
}
checks

# Plots

In [ ]:
def preview_row(axes: np.ndarray, frame: DroneFrame) -> None:
    folder = out / frame.name
    meta = json.loads((folder / "meta.json").read_text())
    old = logged[frame.name]
    w_m, h_m = footprint_m(old.altitude_m, FOV_W_DEG, CAMERA_ASPECT_W_OVER_H)
    panels = [(frame.image(), "drone frame"),
              (satellite.crop_rotated(old.position, old.yaw_deg, w_m, h_m), "orthophoto, logged pose"),
              (Image.open(folder / "satellite.png"), f"reposed: {meta['yaw_offset_deg']:+.0f} deg, x{meta['footprint_scale']:.2f}"),
              (np.asarray(Image.open(folder / "label.png")), "label, reposed")]
    for ax, (image, title) in zip(axes, panels):
        ax.imshow(image, vmin=0, vmax=5) if title.startswith("label") else ax.imshow(image)
        ax.set_title(f"{title}", fontsize=8); ax.axis("off")
    axes[0].set_title(f"{frame.name[-7:]} block {meta['block']}", fontsize=8)

In [ ]:
fig, axes = plt.subplots(len(PREVIEW_FRAMES), 4, figsize=(14, 2.6 * len(PREVIEW_FRAMES)))
for row, name in zip(axes, PREVIEW_FRAMES):
    preview_row(row, next(f for f in reposed if f.name == name))
fig.tight_layout()
fig.savefig(REPO_ROOT / "datasets" / NAME / "preview.png", dpi=90)

# Register

In [ ]:
write_manifest(
    NAME,
    version=VERSION,
    description="The 44 drone-2025-easy frames with heading and footprint scale corrected per frame by "
                "pocs/pose-reregistration; logged position kept. Orthophoto and label crops at that pose.",
    source=f"{SOURCE_FRAMES} + {CORRECTIONS}",
    config={"source_frames": SOURCE_FRAMES, "corrections": CORRECTIONS, "fov_w_deg": FOV_W_DEG,
            "sat_image": SAT_IMAGE, "labels": list(LABELS)},
    samples=len(reposed),
    checks=checks,
)
verify_dataset(NAME)